In [1]:
import pandas as pd
import numpy as np

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
column_names = ["Sex", "Length", "Diameter", "Height", "Whole_weight", 
                "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]
df = pd.read_csv(url, names=column_names)

print("Number of rows:", len(df))
print("Column names:", df.columns.tolist())
print(df.head())

# Checkpoint:
# what is input: The physical measurements of the abalone (Length, Diameter, etc).
# what is output: The age of the abalone (derived from Rings).
# why output is numeric: Because we are predicting a continuous value (age) rather than a discrete class.

Number of rows: 4177
Column names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  


In [2]:
df['y'] = df['Rings'] + 1.5

In [3]:
selected_features = ['Length', 'Diameter', 'Height']
X_all = df[selected_features].values
y_all = df['y'].values

# Justification:
# Feature 1: Length - Organisms typically grow in length as they age.
# Feature 2: Diameter - Width is strictly correlated with overall growth.
# Feature 3: Height - Vertical shell growth is a distinct dimension of aging.

In [4]:
num_samples = len(df)
train_size = int(0.8 * num_samples)
indices = np.arange(num_samples)
np.random.shuffle(indices)
train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train_raw = X_all[train_indices]
y_train = y_all[train_indices].reshape(-1, 1)
X_test_raw = X_all[test_indices]
y_test = y_all[test_indices].reshape(-1, 1)

print("X_train shape:", X_train_raw.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test_raw.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3341, 3)
y_train shape: (3341, 1)
X_test shape: (836, 3)
y_test shape: (836, 1)


In [5]:
mean = np.mean(X_train_raw, axis=0)
std = np.std(X_train_raw, axis=0)

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

# Checkpoint:
# why normalization is needed for learning: It scales features equally so gradients behave stably and the model converges faster.

In [6]:
def forward(X, w, b):
    return np.dot(X, w) + b

d = X_train.shape[1]
w_dummy = np.zeros((d, 1))
b_dummy = 0.0
y_hat_dummy = forward(X_train, w_dummy, b_dummy)

print(f"X shape: {X_train.shape}")
print(f"w shape: {w_dummy.shape}")
print(f"b shape: scalar")
print(f"y_hat shape: {y_hat_dummy.shape}")

# Checkpoint:
# parameters are: The weights (w) and the bias (b).
# number of parameters: 4 (3 weights + 1 bias).

X shape: (3341, 3)
w shape: (3, 1)
b shape: scalar
y_hat shape: (3341, 1)


In [7]:
def mse(y, y_hat):
    N = len(y)
    return (1/N) * np.sum((y - y_hat)**2)

# Checkpoint:
# why square: To ensure errors are positive and to penalize larger errors more heavily.
# what mistakes are expensive: Outliers (large deviations from the true value).

In [8]:
def grad_w(X, y, y_hat):
    N = len(y)
    return (2/N) * np.dot(X.T, (y_hat - y))

def grad_b(y, y_hat):
    N = len(y)
    return (2/N) * np.sum(y_hat - y)

# Checkpoint:
# meaning of large gradient: The model is currently very inaccurate, corresponding to a steep slope in the loss landscape.
# effect of too-large learning rate: The updates will overshoot the minimum, causing divergence or oscillation.
# what gradient means in words: The direction of steepest increase in loss.
# why subtracting gradient reduces loss: Moving opposite to the gradient moves us down the slope (steepest descent).

In [9]:
np.random.seed(42)
w = np.random.randn(3, 1) * 0.01
b = 0.0

learning_rate = 0.01
epochs = 2000

for epoch in range(epochs):
    y_hat = forward(X_train, w, b)
    
    loss = mse(y_train, y_hat)
    
    dw = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)
    
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# Checkpoint:
# Initial expectation: Loss should decrease rapidly then stabilize.
# Revised expectation after training: Confirmed; loss dropped significantly and converged.

Epoch 0, Loss: 140.2924
Epoch 200, Loss: 6.6827
Epoch 400, Loss: 6.6313
Epoch 600, Loss: 6.6224
Epoch 800, Loss: 6.6146
Epoch 1000, Loss: 6.6076
Epoch 1200, Loss: 6.6014
Epoch 1400, Loss: 6.5957
Epoch 1600, Loss: 6.5907
Epoch 1800, Loss: 6.5861


In [10]:
y_test_pred = forward(X_test, w, b)
test_mse = mse(y_test, y_test_pred)
print(f"Test MSE: {test_mse:.4f}")

test_mae = np.mean(np.abs(y_test - y_test_pred))
print(f"Test MAE: {test_mae:.4f}")

print("\n--- 5 Sample Predictions ---")
for i in range(5):
    print(f"True: {y_test[i][0]:.2f}, Pred: {y_test_pred[i][0]:.2f}, Abs Error: {abs(y_test[i][0] - y_test_pred[i][0]):.2f}")

# Checkpoint:
# systematic errors: The model likely underpredicts the age of very old abalones.
# observed bias: Regression to the mean (predictions cluster centrally, missing extremes).

Test MSE: 7.3509
Test MAE: 1.9407

--- 5 Sample Predictions ---
True: 8.50, Pred: 7.57, Abs Error: 0.93
True: 7.50, Pred: 9.01, Abs Error: 1.51
True: 17.50, Pred: 13.37, Abs Error: 4.13
True: 10.50, Pred: 10.49, Abs Error: 0.01
True: 9.50, Pred: 12.12, Abs Error: 2.62
